# 🔬 Notebook 2: Standalone Acoustic Calibration Lab ($k(f)$ Extraction)

Welcome to the **Acoustic Calibration Protocol Suite**. This notebook guides you step-by-step through measuring the physical acoustic constant $k(f)$ across multiple frequencies and distances, validating the inverse-distance law $V_{\text{RMS}} = \frac{k(f)}{r}$ ($R^2 \ge 0.95$), and exporting a portable JSON profile for real-time metric distance tracking.

---

### 🏛️ The Physical Acoustic Principle
Acoustic spherical wave decay dictates:
$$V_{\text{RMS}}(r_i,\, f_j) = k(f_j) \cdot \left(\frac{1}{r_i}\right) + c_{\text{room}}(f_j)$$

* **Slope $k(f_j)$:** The true physical speaker-to-microphone acoustic constant in $\text{Volts}\cdot\text{meters}$.
* **Intercept $c_{\text{room}}(f_j)$:** Absorbs constant ambient room reverberation.
* **Dynamic Boundary Pruner:** Automatically excludes near-field saturation/clipping at small $r$ and far-field room echoes at large $r$, fitting $k(f)$ strictly on the true linear $1/r$ region.
* **Weighted Least Squares (WLS):** Uses statistical variance ($w_i = 1/\sigma_i^2$) across $N=30$ repeat frame observations per station.

## 1. Initialize Hardware Overlay & System Metadata
Instantiate `MicrophoneArrayOverlay` and configure the fixed system metadata (`speaker_volume`, `mic_gain`, `temperature_c`).

In [ ]:
import time
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    AcousticCalibrationProtocol,
    AcousticProfile,
    DistanceEstimator
)

# 1. Initialize hardware
ol = MicrophoneArrayOverlay()

# 2. Fixed System Operating Parameters (Traceability Metadata)
system_metadata = {
    "speaker_device": "Smartphone",
    "speaker_volume_setting": 0.75,          # Fixed at 75% volume slider
    "mic_gain_setting": "+35dB / 12 o clock", # Fixed MAX4466 trimmer pot position
    "environment_label": "Acoustic_Lab_Desk",
    "temperature_c": 20.0
}

# 3. Initialize Calibration Protocol with R^2 >= 0.95 quality gate
protocol = AcousticCalibrationProtocol(r2_threshold=0.95, system_metadata=system_metadata)

print(f"✅ Hardware Overlay Active: {ol.current_profile} mode ({ol.fs_per_ch:.0f} SPS)")
print(f"✅ System Metadata Locked: Volume={system_metadata['speaker_volume_setting']*100:.0f}%, Gain={system_metadata['mic_gain_setting']}")
print(f"✅ Calibration Protocol Initialized (R² Quality Gate >= {protocol.r2_threshold})")

## 2. Define Master High-Density Calibration Grid
Defines the **14 distance stations** ($0.15\,\text{m} - 1.00\,\text{m}$) and **26 bin-aligned carrier frequencies** ($390.6\,\text{Hz} - 6005.8\,\text{Hz}$).

In [ ]:
# 14 Distance Stations in meters (Dense near-to-mid coverage)
calib_distances_m = [
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.70, 0.80, 0.90, 1.00
]

# 26 Exact Bin-Aligned Carrier Frequencies (k_bin * 24.4140625 Hz)
calib_frequencies_hz = [
    # 1. Smooth Low Band (150-200 Hz steps)
    390.6,   512.7,   683.6,   854.5,  1000.9,  1220.7,  1440.4,
    # 2. Dense Resonance & Transition Zone (50-100 Hz steps)
    1611.3,  1708.9,  1806.6,  1904.3,  2001.9,  2124.0,  2246.1,
    2368.1,  2489.2,  2612.3,  2734.4,  2856.4,  3002.9,  3222.6,  3442.4,
    # 3. High Band (300-500 Hz steps)
    3808.6,  4296.9,  5004.9,  6005.8
]

print(f"Master Calibration Grid Configured:")
print(f"  • Distance Stations   ({len(calib_distances_m)} stations)   : {calib_distances_m} meters")
print(f"  • Frequency Carriers  ({len(calib_frequencies_hz)} tones)      : {calib_frequencies_hz[0]:.1f} Hz -> {calib_frequencies_hz[-1]:.1f} Hz")
print(f"  • Total Calibration Observations  : {len(calib_distances_m) * len(calib_frequencies_hz)} grid cells (N=30 samples/cell)")

## 3. Station-Based Burst Statistical Capture ($N=30$ frames per tone)
For each distance station, place the phone at the tape mark, play the stepped tones, and run `capture_station_sweep()`.

In [ ]:
# Statistical Batch Sweep Capture Function
def capture_station_sweep(distance_m, target_frequencies=None, n_samples=30):
    """
    Captures N=30 burst frames across all frequencies at a fixed distance station,
    computing sample mean, standard deviation, and statistical confidence intervals.
    """
    freqs_to_run = target_frequencies if target_frequencies is not None else calib_frequencies_hz
    print(f"\n📍 Capturing Station at r = {distance_m:.2f} meters ({len(freqs_to_run)} tones, N={n_samples} samples/tone)...")
    station_results = {}
    
    for f_target in freqs_to_run:
        samples = []
        for _ in range(n_samples):
            # Hybrid dual-DMA capture (BFP-immune)
            frame = ol.capture_quadruple(source="A0", f_min=max(20.0, f_target-40.0), f_max=f_target+40.0, timeout=0.35)
            q = frame["quadruple"]
            if q["amplitude_v"] > 0.0005:  # Valid audio frame
                samples.append(q["amplitude_v"])
            time.sleep(0.005)
        
        if len(samples) > 0:
            protocol.add_measurement(distance_m=distance_m, frequency_hz=f_target, amplitude_v=samples)
            mean_v = float(np.mean(samples))
            std_v = float(np.std(samples, ddof=1)) if len(samples) > 1 else 0.0
            sem_v = std_v / np.sqrt(len(samples)) if len(samples) > 0 else 0.0
            station_results[f_target] = {"mean_v": mean_v, "std_v": std_v, "sem_v": sem_v, "n": len(samples)}
            print(f"    • f={f_target:6.1f} Hz: V_RMS = {mean_v*1000.0:6.2f} mV ± {sem_v*1000.0:4.2f} mV (σ={std_v*1000.0:4.2f} mV, N={len(samples)})")
        else:
            print(f"    ⚠ f={f_target:6.1f} Hz: Skipped (no tone detected)")
            
    return station_results

print("Ready for guided calibration. (Call capture_station_sweep(distance_m) at each station mark).")

## 4. WLS Regression, Dynamic Pruning & Interactive Diagnostics
Solves the Weighted Least Squares regressions over the automatically pruned linear $1/r$ sub-windows and displays:
1. **$V_{\text{RMS}}$ vs $1/r$ Linear Curves** with $\pm 1\sigma$ error bars and visual demarcation of pruned vs. active points.
2. **Continuous $k(f)$ Function** [$\text{V}\cdot\text{m}$].
3. **Goodness-of-Fit $R^2(f)$ Score** against the $0.95$ Quality Gate.

In [ ]:
# Execute WLS Regression & Dynamic Boundary Pruning
fit_results = protocol.fit()

# -----------------------------------------------------------------------------
# Plotly Interactive Diagnostic Dashboard
# -----------------------------------------------------------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "<b>1. WLS Regressions (Error Bars & Pruning)</b>",
        "<b>2. Calibrated k(f) Function [V·m]</b>",
        "<b>3. Goodness-of-Fit R²(f) Quality Gate</b>"
    ),
    horizontal_spacing=0.08
)

colors = ["#00FFCC", "#FFA500", "#00E5FF", "#FF007F", "#76FF03", "#E040FB", "#FFD600", "#00E676"]

valid_freqs = sorted(fit_results.keys())
k_list = [fit_results[f]["k"] for f in valid_freqs]
k_err_list = [fit_results[f]["delta_k"] for f in valid_freqs]
r2_list = [fit_results[f]["r_squared"] for f in valid_freqs]

for idx, f in enumerate(valid_freqs):
    res = fit_results[f]
    stations = res["raw_stations"]
    
    r_active = np.array([st["r_m"] for st in stations if st["is_pruned_in"]])
    v_active = np.array([st["mean_v"] for st in stations if st["is_pruned_in"]]) * 1000.0
    err_active = np.array([st["std_v"] for st in stations if st["is_pruned_in"]]) * 1000.0
    
    r_pruned = np.array([st["r_m"] for st in stations if not st["is_pruned_in"]])
    v_pruned = np.array([st["mean_v"] for st in stations if not st["is_pruned_in"]]) * 1000.0
    
    c = colors[idx % len(colors)]
    
    # Active linear points with error bars
    if len(r_active) > 0:
        inv_r_active = 1.0 / r_active
        fig.add_scatter(
            x=inv_r_active, y=v_active,
            error_y=dict(type="data", array=err_active, visible=True),
            mode="markers", marker=dict(size=8, color=c),
            name=f"{f:.0f} Hz (Active)", row=1, col=1
        )
        
        # Fitted regression line
        inv_r_line = np.linspace(min(inv_r_active)*0.9, max(inv_r_active)*1.1, 40)
        v_line = (res["k"] * inv_r_line + res["c_room"]) * 1000.0
        fig.add_scatter(
            x=inv_r_line, y=v_line, mode="lines", line=dict(color=c, dash="dash"),
            name=f"{f:.0f} Hz (Fit)", row=1, col=1
        )
    
    # Pruned extreme points (grayed out)
    if len(r_pruned) > 0:
        fig.add_scatter(
            x=1.0 / r_pruned, y=v_pruned, mode="markers",
            marker=dict(size=6, color="gray", symbol="x"),
            name=f"{f:.0f} Hz (Pruned)", row=1, col=1,
            showlegend=False
        )

# Column 2: Continuous k(f) function with error bars
fig.add_scatter(
    x=valid_freqs, y=k_list,
    error_y=dict(type="data", array=k_err_list, visible=True),
    mode="lines+markers", line=dict(color="#00FFCC", width=2.5), marker=dict(size=8),
    name="k(f)", row=1, col=2
)

# Column 3: R^2 Quality Bar Chart
fig.add_bar(x=[f"{f:.0f}" for f in valid_freqs], y=r2_list, marker=dict(color="#00E5FF"), name="R² Score", row=1, col=3)
fig.add_hline(y=protocol.r2_threshold, line=dict(color="orange", dash="dash"), annotation_text=f"R² Gate ({protocol.r2_threshold})", row=1, col=3)

fig.update_layout(template="plotly_dark", height=460, title="<b>Acoustic Calibration Laboratory: WLS Regressions & Quality Analysis</b>")
fig.update_xaxes(title="1 / Distance [m⁻¹]", row=1, col=1)
fig.update_yaxes(title="V_RMS [mV]", row=1, col=1)
fig.update_xaxes(title="Frequency [Hz]", row=1, col=2)
fig.update_yaxes(title="k [V·m]", row=1, col=2)
fig.update_xaxes(title="Frequency [Hz]", row=1, col=3)
fig.update_yaxes(title="R² Score", range=[0.85, 1.01], row=1, col=3)

fig.show()

## 5. Export Calibrated Profile with Metadata & Operational Bounds
Saves the profile to a portable JSON file preserving the system metadata and certified operational bounds.

In [ ]:
export_path = "calibrated_room_profile.json"
saved_file = protocol.save_profile_json(
    filepath=export_path,
    name="Smartphone_Lab_WLS_Profile",
    description="WLS calibrated 1/r profile with dynamic boundary pruning and N=30 statistical variance"
)

print(f"🎉 SUCCESS: Saved calibrated profile to: {saved_file.resolve()}")

## 6. Live Distance Inversion & Operational Boundary Verification
Loads the newly created profile into `DistanceEstimator` and tracks physical distance in centimeters with certified boundary checking.

In [ ]:
# Load newly generated profile into DistanceEstimator
loaded_profile = AcousticProfile.from_json(export_path)
estimator = DistanceEstimator(profile=loaded_profile, noise_gate_v=0.003)

# Capture live frame and test metric distance inversion
frame = ol.capture_quadruple(source="A0", f_min=100.0, f_max=15000.0, timeout=0.5)
result = estimator.process_frame(frame, source="A0")

print("\n--- Live Inverted Distance Result ---")
print(f"Detected Pitch f0  : {result['frequency_hz']:.1f} Hz")
print(f"In-Band Amplitude  : {result['amplitude_v']*1000:.2f} mV")
print(f"Evaluated k(f0)    : {result['k_evaluated']:.4f} V·m")
print(f"Inverted Distance  : {result['distance_m']*100:.1f} cm (±{result['distance_err_m']*100:.1f} cm)")
print(f"Operational Status : {result['distance_status']}")

ol.close()
print("\n🔒 Hardware resources cleanly released.")